<a href="https://colab.research.google.com/github/vidit-demog/qlearning-regional-exposure-routing/blob/main/penn_homi_net_data_man_share.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
# Libraries
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import itertools
import time
import os

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/My Drive/countyhomicdc.csv')
display(df)

# Convert Rate to numeric, coerce errors to NaN
df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')


,GEOID,NAME,ST_GEOID,ST_NAME,Intent,Period,Count,Rate,Rate_M,Rate_M_CI,Data_As_Of,TTM_Date_Range
0,27091,Martin County,27,Minnesota,All_Suicide,2021,1-9,14.4,1,9.7-22.1,12/11/2025,NaN
1,27091,Martin County,27,Minnesota,FA_Suicide,2023,1-9,7.0,1,4.1-12.4,12/11/2025,NaN
2,27091,Martin County,27,Minnesota,FA_Suicide,2024,1-9,10.0,1,6.1-17.4,12/11/2025,NaN
3,27093,Meeker County,27,Minnesota,Drug_OD,2022,1-9,9.4,1,4.9-19.1,12/11/2025,NaN
4,27095,Mille Lacs County,27,Minnesota,Drug_OD,2021,15,55.7,0,NaN,12/11/2025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
131995,31149,Rock County,31,Nebraska,FA_Suicide,2022,0,0.0,0,NaN,12/11/2025,NaN
131996,31151,Saline County,31,Nebraska,All_Homicide,2022,0,0.0,0,NaN,12/11/2025,NaN
131997,31151,Saline County,31,Nebraska,All_Homicide,TTM,0,0.0,0,NaN,12/11/2025,"August, 2024 to July, 2025"
131998,31151,Saline County,31,Nebraska,All_Suicide,2022,1-9,15.9,1,10.9-23.9,12/11/2025,NaN


In [4]:
filtered_df = df[(df['ST_NAME'] == 'Pennsylvania') & (df['Intent'] == 'All_Homicide') & (df['Period']=='2024')]
display(filtered_df)

,GEOID,NAME,ST_GEOID,ST_NAME,Intent,Period,Count,Rate,Rate_M,Rate_M_CI,Data_As_Of,TTM_Date_Range
5868,42043,Dauphin County,42,Pennsylvania,All_Homicide,2024,25,8.6,0,NaN,12/11/2025,NaN
5945,42113,Sullivan County,42,Pennsylvania,All_Homicide,2024,0,0.0,0,NaN,12/11/2025,NaN
8454,42037,Columbia County,42,Pennsylvania,All_Homicide,2024,0,0.0,0,NaN,12/11/2025,NaN
8494,42059,Greene County,42,Pennsylvania,All_Homicide,2024,0,0.0,0,NaN,12/11/2025,NaN
8517,42077,Lehigh County,42,Pennsylvania,All_Homicide,2024,1-9,2.1,1,1.3-4.1,12/11/2025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
120869,42105,Potter County,42,Pennsylvania,All_Homicide,2024,1-9,-999.0,1,-999,12/11/2025,NaN
121487,42061,Huntingdon County,42,Pennsylvania,All_Homicide,2024,0,0.0,0,NaN,12/11/2025,NaN
121506,42075,Lebanon County,42,Pennsylvania,All_Homicide,2024,1-9,2.4,1,1.2-5.8,12/11/2025,NaN
121550,42111,Somerset County,42,Pennsylvania,All_Homicide,2024,1-9,2.3,1,1.0-6.4,12/11/2025,NaN


In [6]:
# Create dataframe
df2 = filtered_df[['GEOID', 'NAME', 'Rate', 'Rate_M']].copy()

# Clean NAME
df2['NAME'] = (
    df2['NAME']
    .str.replace(r'\s*county\s*$', '', regex=True, case=False)
    .str.strip()
)

# Rename NAME column to county
df2 = df2.rename(columns={'NAME': 'County'})

replacement = df2.loc[df2['Rate'] >= 0, 'Rate'].median()
df2.loc[df2['Rate'] == -999, 'Rate'] = replacement

# RScore generation to make a more workable risk based metric for later
valid_rates = df2.loc[df2['Rate'] >= 0, 'Rate']
median_rate = valid_rates.median()


df2['RScore'] = (df2['Rate'] / median_rate) * 10
df2['RScore'] = df2['RScore'].clip(lower=0) # Safety check: no negative RScores
df2['RScore'] = df2['RScore'].round().astype(int) # Make RScores whole numbers

# Save to Google Drive
out_path = "/content/drive/MyDrive/df2_rscore.csv"
df2.to_csv(out_path, index=False)


In [7]:
OUT_PATH = "/content/drive/My Drive/pa_county_adjacency_distances.csv"

# Load counties

counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
)

pa = counties[counties["STATEFP"] == "42"].to_crs(epsg=4326)

# Fix potential geometry issues
pa["geometry"] = pa.geometry.buffer(0)

# Centroids
pa["lon"] = pa.geometry.centroid.x
pa["lat"] = pa.geometry.centroid.y

pa = pa[["NAME", "geometry", "lon", "lat"]].reset_index(drop=True)

# Build adjacency list (shared borders only)

adj = gpd.sjoin(
    pa[["NAME", "geometry"]],
    pa[["NAME", "geometry"]],
    predicate="touches",
    how="inner"
)

# Remove self-loops (county touching itself) and ensure unique pairs
adj = adj[adj["NAME_left"] != adj["NAME_right"]] # Remove self-loops

# Unique identifier for each pair (e.g., sort county names)
# To ensure that ('A', 'B') and ('B', 'A') resolve to the same identifier
adj['pair_id'] = adj.apply(lambda row: tuple(sorted((row['NAME_left'], row['NAME_right']))), axis=1)

# Drop duplicates based on this pair_id, keeping only one instance of each unique pair
adj = adj.drop_duplicates(subset=['pair_id'])

# Drop the temporary 'pair_id' column as it's no longer needed
adj = adj.drop(columns=['pair_id'])

adj = adj.rename(columns={
    "NAME_left": "county1",
    "NAME_right": "county2"
})[["county1", "county2"]]

# Resume if file exists
# Reset file (important to avoid contamination)
if os.path.exists(OUT_PATH):
    os.remove(OUT_PATH)
    print("Old file removed — starting fresh")

if os.path.exists(OUT_PATH):
    distance_df = pd.read_csv(OUT_PATH)
    done_pairs = set(zip(distance_df.county1, distance_df.county2))
    print(f"Resuming: {len(done_pairs)} edges already computed")
elif not adj.empty: # Only initialize if adj is not empty, otherwise it can cause issues with columns
    distance_df = pd.DataFrame(columns=adj.columns.tolist() + ["distance_km"])
    done_pairs = set()
else: # if adj is empty, distance_df should be empty too
    distance_df = pd.DataFrame(columns=["county1", "county2", "distance_km"])
    done_pairs = set()

rows = []

# Query OSRM ONLY for adjacent counties

for _, row in adj.iterrows():
    pair = (row.county1, row.county2)
    if pair in done_pairs:
        continue

    c1 = pa.loc[pa.NAME == row.county1].iloc[0]
    c2 = pa.loc[pa.NAME == row.county2].iloc[0]

    url = (
        f"http://router.project-osrm.org/route/v1/driving/"
        f"{c1.lon},{c1.lat};{c2.lon},{c2.lat}?overview=false"
    )

    try:
        r = requests.get(url, timeout=10).json()
        dist_km = r["routes"][0]["distance"] / 1000

        rows.append({
            "county1": row.county1,
            "county2": row.county2,
            "distance_km": dist_km
        })

    except Exception:
        print(f"Failed: {pair}")
        continue

    # Save every 25 edges
    if len(rows) % 25 == 0:
        distance_df = pd.concat([distance_df, pd.DataFrame(rows)], ignore_index=True)
        distance_df.to_csv(OUT_PATH, index=False)
        rows = []
        print(f"Saved {len(distance_df)} edges")

    time.sleep(0.25)

# Final save

if rows:
    distance_df = pd.concat([distance_df, pd.DataFrame(rows)], ignore_index=True)
    distance_df.to_csv(OUT_PATH, index=False)

print("Finished and saved county adjacency network")

/tmp/ipykernel_2918/1986296426.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pa["lon"] = pa.geometry.centroid.x
/tmp/ipykernel_2918/1986296426.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pa["lat"] = pa.geometry.centroid.y


Old file removed — starting fresh


/tmp/ipykernel_2918/1986296426.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  distance_df = pd.concat([distance_df, pd.DataFrame(rows)], ignore_index=True)


Saved 25 edges
Saved 50 edges
Saved 75 edges
Saved 100 edges
Saved 125 edges
Saved 150 edges
Finished and saved county adjacency network


In [8]:
county_coords = pa[["NAME", "lon", "lat"]].rename(columns={"NAME": "County"})

county_coords.to_csv(
    "/content/drive/My Drive/pa_county_centroids.csv",
    index=False
)


### Replicating RScore calculation, Adjacency Network, and Centroid Saving for Pennsylvania + New Jersey

In [9]:
OUT_PATH_ADJACENCY = "/content/drive/My Drive/pa_nj_county_adjacency_distances.csv"
OUT_PATH_RISK = "/content/drive/My Drive/df_pa_nj_rscore.csv"
OUT_PATH_CENTROIDS = "/content/drive/My Drive/pa_nj_county_centroids.csv"

for p in [OUT_PATH_ADJACENCY, OUT_PATH_RISK, OUT_PATH_CENTROIDS]:
    if os.path.exists(p):
        os.remove(p)

filtered_df_pa_nj = df[
    (df['ST_NAME'].isin(['Pennsylvania', 'New Jersey'])) &
    (df['Intent'] == 'All_Homicide') &
    (df['Period'] == '2024')
].copy()

df_pa_nj_rscore = filtered_df_pa_nj[['GEOID', 'NAME', 'Rate', 'Rate_M']].copy()

df_pa_nj_rscore['County'] = (
    df_pa_nj_rscore['NAME']
    .str.replace(r'\s*county\s*$', '', regex=True, case=False)
    .str.strip()
)

df_pa_nj_rscore = df_pa_nj_rscore.drop(columns=['NAME'])

df_pa_nj_rscore.loc[df_pa_nj_rscore['Rate'] == -999, 'Rate'] = \
    df_pa_nj_rscore.loc[df_pa_nj_rscore['Rate'] >= 0, 'Rate'].median()

median_rate = df_pa_nj_rscore.loc[df_pa_nj_rscore['Rate'] >= 0, 'Rate'].median()

df_pa_nj_rscore['RScore'] = (df_pa_nj_rscore['Rate'] / median_rate) * 10
df_pa_nj_rscore['RScore'] = df_pa_nj_rscore['RScore'].clip(lower=0).round().astype(int)

df_pa_nj_rscore.to_csv(OUT_PATH_RISK, index=False)

counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
)

pa = counties[counties["STATEFP"] == "42"].copy()
nj = counties[counties["STATEFP"] == "34"].copy()

combined_gdf = pd.concat([pa, nj]).copy()

combined_gdf["county_id"] = combined_gdf["STATEFP"] + "_" + combined_gdf["COUNTYFP"]

combined_gdf["geometry"] = combined_gdf.geometry.buffer(0)

combined_proj = combined_gdf.to_crs(epsg=3857)

combined_proj["centroid"] = combined_proj.geometry.centroid
combined_proj["lon"] = combined_proj.centroid.to_crs(epsg=4326).x
combined_proj["lat"] = combined_proj.centroid.to_crs(epsg=4326).y

adj_raw = gpd.sjoin(
    combined_proj[["county_id", "geometry"]],
    combined_proj[["county_id", "geometry"]],
    predicate="touches",
    how="inner"
)

adj_raw = adj_raw[adj_raw["county_id_left"] != adj_raw["county_id_right"]]

adj_raw["pair_id"] = adj_raw.apply(
    lambda r: tuple(sorted((r["county_id_left"], r["county_id_right"]))),
    axis=1
)

adj_unique = adj_raw.drop_duplicates("pair_id")

proj_indexed = combined_proj.set_index("county_id")

valid_pairs = []

for _, r in adj_unique.iterrows():
    g1 = proj_indexed.loc[r["county_id_left"]].geometry
    g2 = proj_indexed.loc[r["county_id_right"]].geometry

    inter = g1.intersection(g2)

    if not inter.is_empty:
        valid_pairs.append({
            "county1": r["county_id_left"],
            "county2": r["county_id_right"]
        })

adj_pa_nj = pd.DataFrame(valid_pairs)

coord_lookup = combined_proj.set_index("county_id")[["lon", "lat"]].to_dict("index")

rows = []

for _, r in adj_pa_nj.iterrows():

    c1 = coord_lookup[r["county1"]]
    c2 = coord_lookup[r["county2"]]

    url = f"http://router.project-osrm.org/route/v1/driving/{c1['lon']},{c1['lat']};{c2['lon']},{c2['lat']}?overview=false"

    try:
        res = requests.get(url, timeout=10).json()

        if "routes" in res:
            rows.append({
                "county1": r["county1"],
                "county2": r["county2"],
                "distance_km": res["routes"][0]["distance"] / 1000
            })

    except:
        continue

    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(OUT_PATH_ADJACENCY, index=False)

    time.sleep(0.2)

distance_df_pa_nj = pd.DataFrame(rows)
distance_df_pa_nj.to_csv(OUT_PATH_ADJACENCY, index=False)

centroids = combined_proj[["county_id", "lon", "lat"]].copy()
centroids.to_csv(OUT_PATH_CENTROIDS, index=False)